# Comparación de Modelos

In [9]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.scorer import VolatilityScorer
from src.utils import test_toolkit as ts
from src.utils import ts_toolkit as tst
from src.utils import trainer_toolkit as tt

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results"

csv_paths = [
    RESULTS_PATH / "garch" / "best_garch_predictions.csv",
    RESULTS_PATH / "egarch" / "best_egarch_predictions.csv",
    RESULTS_PATH / "har" / "best_har_predictions.csv",
    RESULTS_PATH / "wtd_har" / "best_wtd_har_predictions.csv",
    RESULTS_PATH / "gru" / "exec_2" / "gru_test_predictions.csv",
    RESULTS_PATH / "tcn" / "exec_6" / "tcn_test_predictions.csv",
    RESULTS_PATH / "markov_tcn" / "exec_7" / "markov_tcn_test_predictions.csv",
    RESULTS_PATH / "bigru_tcn" / "exec_1" / "bigru_tcn_test_predictions.csv",
    RESULTS_PATH / "causal_transformer" / "exec_2" / "causal_transformer_test_predictions.csv",
    RESULTS_PATH / "conv_transformer" / "exec_1" / "conv_transformer_test_predictions.csv",
    RESULTS_PATH / "band_causal_tcn" / "exec_1" / "band_causal_tcn_test_predictions.csv",
]

model_names = [
    "GARCH",
    "EGARCH",
    "HAR",
    "WTD-HAR",
    "GRU",
    "TCN",
    "Markov TCN",
    "BiGRU-TCN",
    "Cau. Trans",
    "Conv. Trans",
    "Band TCN",
]

test_bundle = ts.load_test_predictions_matrix(
    csv_paths=csv_paths,
    model_names=model_names,
)

dates_test = test_bundle["dates"]
y_true_test = test_bundle["y_true"]
preds_matrix = test_bundle["preds_matrix"]
models_loaded = test_bundle["model_names"]

## Introducción

Esta libreta se dedica a la comparación conjunta del desempeño predictivo fuera de muestra de los modelos entrenados y evaluados para pronosticar la varianza realizada de Rogers-Satchell (RS) de los futuros del crudo Brent (BZ=F) bajo el protocolo del benchmark. A diferencia de las libretas anteriores, donde cada modelo se formuló, calibró y diagnosticó de manera individual tanto dentro como fuera de la muestra, aquí se alinean temporalmente las series pronosticadas sobre el conjunto de prueba para contrastar directamente las distintas arquitecturas bajo las mismas condiciones de información.

El análisis integra métricas puntuales y probabilísticas de evaluación, pruebas de comparación predictiva por pares mediante la prueba de Diebold-Mariano con bootstrap de bloques estacionarios, medidas de magnitud del efecto a través del delta de Cliff y procedimientos de selección del conjunto superior de modelos (Model Confidence Set) sobre la pérdida QLIKE. Este marco comparativo permite determinar si las diferencias de desempeño entre modelos reflejan ventajas predictivas consistentes o variaciones asociadas a los regímenes de volatilidad del mercado.

## Resultados principales

En la comparación fuera de muestra, TCN encabezó el desempeño en la métrica principal (QLIKE), en las métricas heterogéneas (HMSE y HMAE) y en CRPS, seguido de cerca en QLIKE por Causal Transformer y Band Causal TCN para conformar el grupo de los tres primeros modelos. Estos tres enfoques mantuvieron valores bajos y competitivos tanto en las pérdidas cuantilares de Pinball como en RMSE, superando en QLIKE y HMSE a referencias econométricas como WTD-HAR, GARCH y HAR, y marcando una ventaja clara frente a EGARCH, GRU y BiGRU-TCN, donde se observaron incrementos pronunciados en QLIKE y HMSE.

In [4]:
ts.display_models_comparison(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metrics=("qlike", "crps", "pinball_05", "pinball_95", "hmse", "hmae"),
    sort_by="qlike",
    decimals=4,
)

Modelo,QLIKE,CRPS,Pinball (τ=0.05),Pinball (τ=0.95),HMSE,HMAE
TCN,0.3675,0.0123,0.0027,0.0015,1.7030,0.6482
Cau. Trans,0.3936,0.0123,0.0027,0.0016,2.1469,0.6853
Band TCN,0.4022,0.0124,0.0027,0.0016,2.0900,0.6927
Markov TCN,0.4026,0.0123,0.0027,0.0016,2.3603,0.6904
WTD-HAR,0.4084,0.0124,0.0026,0.0015,3.1898,0.7317
GARCH,0.4097,0.0123,0.0028,0.0016,2.5417,0.6710
HAR,0.4471,0.0124,0.0026,0.0016,3.3125,0.7970
Conv. Trans,0.4793,0.0124,0.0026,0.0016,5.2286,0.8457
EGARCH,0.5068,0.0123,0.0028,0.0017,10.1008,0.7719
GRU,0.5109,0.0124,0.0027,0.0017,6.4425,0.8131


Para evaluar la significancia estadística y la magnitud del efecto entre pares de modelos sobre la pérdida QLIKE, se aplicó la prueba de Diebold-Mariano con bootstrap de bloques ($\alpha = 0.05$) y el delta de Cliff. Este procedimiento permite verificar si los intervalos de confianza bootstrap al 95 % excluyen el cero y cuantificar la dirección y magnitud ordinal de cada diferencia por pares.

Considerando que TCN obtuvo el menor QLIKE medio fuera de muestra, su ventaja frente a Causal Transformer y Band Causal TCN muestra un matiz clave al contrastar ambos criterios. En la fila de Causal Transformer frente a TCN se observa $\delta = +0.03$, y en la fila de Band Causal TCN frente a TCN se observa $\delta = +0.02$, lo que indica una ligera tendencia a mayor pérdida relativa respecto a TCN con tamaños de efecto reducidos en términos de magnitud del delta de Cliff. Entre Causal Transformer y Band Causal TCN aparece un punto negro (DM no significativo, intervalo al 95 % que incluye el cero), por lo que no se encuentra evidencia de una diferencia estadísticamente significativa entre ambos.

In [12]:
dm_results = ts.diebold_mariano_test(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metric="qlike",
    alpha=0.05,
    n_bootstrap=2000,
    seed=SEED,
)

cliff_results = ts.cliffs_delta(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metric="qlike",
)

ts.plot_cliffs_dm_heatmap(
    cliff_results=cliff_results,
    dm_results=dm_results,
    model_order=models_loaded,
    title="Delta de Cliff por Pares (Referencia DM — QLIKE)",
    fmt="svg",
)

Más allá de los tres primeros puestos, TCN presenta diferencias estadísticamente significativas y de magnitud pequeña o moderada en su escala frente a varios modelos con pérdidas superiores, como frente a EGARCH ($\delta = -0.08$), GRU ($\delta = -0.07$) y GARCH ($\delta = -0.06$), mientras que frente a WTD-HAR el intervalo de Diebold-Mariano incluye el cero. En conjunto, TCN lidera el desempeño puntual, pero la distancia frente a Causal Transformer y Band Causal TCN es acotada en magnitud ordinal y sin separación estadísticamente significativa entre estos dos últimos, lo que sugiere un desempeño cercano en el tramo superior del benchmark.

El método de simulación histórica filtrada (*Filtered Historical Simulation*, FHS) construye bandas de predicción para los retornos y el precio del crudo Brent escalando cuantiles empíricos mediante la volatilidad condicional pronosticada por el modelo $\left(\hat{\sigma}_t = \sqrt{\widehat{RS}_t}\right)$. A partir de los residuos estandarizados obtenidos al dividir los retornos observados entre la volatilidad estimada, se extraen los cuantiles empíricos correspondientes al nivel de confianza deseado y se proyectan sobre los retornos futuros, para luego transformarlos a niveles de precio mediante la relación exponencial $P_t = P_{t-1}\exp(r_t)$. Este enfoque semiparamétrico evita imponer una distribución cerrada como la normal, captura asimetrías y colas pesadas presentes en los datos históricos y adapta el ancho de la banda de forma dinámica según el régimen de volatilidad.

En el gráfico de bandas de predicción para el modelo TCN, la banda sombreada acompaña la trayectoria del precio real ampliándose cuando aumenta la volatilidad pronosticada y estrechándose en tramos de mayor calma. De este modo, el intervalo se ajusta de manera dinámica a los cambios en la dispersión esperada del mercado, permitiendo evaluar visualmente cuándo el precio realizado permanece dentro del corredor proyectado y en qué episodios puntuales los movimientos bruscos superan los límites estimados.

In [8]:
DATA_PATH = PROJECT_ROOT / "data" / "processed"
df_test = pd.read_parquet(DATA_PATH / "brent_test.parquet")
df_test.index = pd.to_datetime(df_test.index)

close_col = next(
    (c for c in df_test.columns if c.lower() in {"close", "adj_close", "adj close", "price"}),
    None,
)
if close_col is None:
    raise KeyError(f"No se encontró columna de cierre en brent_test.parquet: {list(df_test.columns)}")

tcn_col_idx = models_loaded.index("TCN")
var_pred_tcn = preds_matrix[:, tcn_col_idx]
prices_aligned = df_test.reindex(dates_test)[close_col].to_numpy(dtype=float)

df_tcn_bands = ts.plot_price_prediction_bands(
    dates=dates_test,
    prices_true=prices_aligned,
    pred_volatility=var_pred_tcn,
    is_variance=True,
    alpha=0.05,
    method="fhs",
    title="Bandas de Predicción en Precio (TCN)",
)